# SIBDA-1380 – Avance 2: Analítica, Clasificación, Inteligencia Artificial, Gestión de Telemercaderistas, KPIs y Gobierno de Datos
**Proyecto:** Analítica de Benefactores y Donaciones – Fundación 1380  
**Asignatura:** ET0155 – Fundamentos de Big Data  
**Entregable:** Avance 2 (Notebook completo y ejecutable en Google Colab)

### Contenido Integrado de este Notebook (Cumplimiento 100 % Avance 1 + Avance 2)
1. **Módulo 1 – ETL Mejorado y Funcional:** Carga inmutable con verificación SHA-256, desambiguación de las 130 columnas, normalización de fechas/categorías, extracción NLP/Regex sobre texto libre (vínculos familiares, restricciones de no contacto, canales) y *unpivot* de los 19 años de historia (`20.804` transacciones por `COP $3.020.016.428`).
2. **Módulo 2 – Estructura de Almacenamiento (Data Lake + Data Warehouse):** Capas `raw`, `bronze`, `silver` y `gold` en **Parquet** + modelo en estrella en **DuckDB** (`dim_benefactor`, `dim_telemercaderista`, `dim_recurso`, `dim_tiempo`, `fact_aportes`, `fact_agenda_contactos`, `audit_log_etl`).
3. **Módulo 3 – Clasificación y Comportamiento de Benefactores:** Clasificación analítica en 4 segmentos (*Permanente o recurrente*, *Esporádico*, *Único aporte registrado*, *Sin aportes recientes que requieren revisión*), cálculo de los 8 atributos individuales, detección de aportes con familiares/amigos (`530` casos) y buscador de ficha individual.
4. **Módulo 4 – Inteligencia Artificial y Predicción de Afinidad:** Modelo supervisado (`RandomForestClassifier` vs `LogisticRegression` en `scikit-learn`) + motor híbrido RFM/NLP para los **9 recursos de captación (`a`–`i`)**, evaluación completa (Accuracy, Precision, Recall, F1, ROC-AUC, Matriz de Confusión, Curva ROC, Feature Importance) y **prototipo interactivo de recomendación** por Telemercaderista, Fecha de gestión y Recurso.
5. **Módulo 5 – Gestión de Telemercaderistas y Control de Teléfonos Compartidos:** Agenda con los 9 campos requeridos, soporte para Viviana, Sandra, Lina y nuevos usuarios, bloqueo anti-saturación y ejecución del **Caso Obligatorio de Prueba: los 4 hermanos (Juan, Felipe, Jesús y Daniel)** más grupos reales del Excel.
6. **Módulo 6 – Indicadores (KPIs), Tablero de Control y Data Storytelling:** Cálculo y visualización de los 11 indicadores mínimos exigidos.
7. **Módulo 8 – Avances en Seguridad y Gobierno de Datos:** Índice de calidad, roles de gobierno, control de acceso por rol (RBAC), protección de datos sensibles (`ENFERMO`/`CALAMIDAD`), backups automatizados, linaje (`audit_log_etl`) y verificación final de inmutabilidad del Excel fuente.

In [ ]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
!pip install -q duckdb openpyxl pyarrow scikit-learn plotly

import os
import re
import uuid
import hashlib
import shutil
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
import duckdb
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, confusion_matrix, roc_curve, classification_report
)
import hmac

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Librerías cargadas correctamente | Pandas:", pd.__version__, "| DuckDB:", duckdb.__version__)

## 2. Módulo 1 – ETL Mejorado y Funcional (Ingesta Inmutable, Limpieza, NLP/Regex y Unpivot)
Garantiza que el archivo Excel original **nunca sea modificado**, verificando su huella criptográfica **SHA-256** antes y después del procesamiento.

In [ ]:
# ==============================================================================
# 2.1 INGESTA SEGURA E INMUTABLE (DATA LAKE /RAW)
# ==============================================================================
BASE_DIR = Path("sibda_1380")
RAW_DIR = BASE_DIR / "datalake" / "raw"
BRONZE_DIR = BASE_DIR / "datalake" / "bronze"
SILVER_DIR = BASE_DIR / "datalake" / "silver"
GOLD_DIR = BASE_DIR / "datalake" / "gold"
DW_DIR = BASE_DIR / "warehouse"
BACKUP_DIR = BASE_DIR / "backups"
LOGS_DIR = BASE_DIR / "logs"

for d in [RAW_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR, DW_DIR, BACKUP_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NOMBRE_ESPERADO = "BD_F1380_SeudonimizadaAnonimizadaencriptada.xlsx"
RUTA_EXCEL_DRIVE = f"/content/drive/MyDrive/Fundacion1380/{NOMBRE_ESPERADO}"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

origen_path = None
for p in [Path(NOMBRE_ESPERADO), Path("Recursos") / NOMBRE_ESPERADO, Path("../Recursos") / NOMBRE_ESPERADO, Path(RUTA_EXCEL_DRIVE)]:
    if p.exists():
        origen_path = p
        break

if origen_path is None:
    try:
        from google.colab import files
        print(f"Sube el archivo '{NOMBRE_ESPERADO}':")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No se subió ningún archivo.")
        origen_path = Path(list(uploaded.keys())[0])
    except ImportError:
        raise FileNotFoundError(f"Ubica '{NOMBRE_ESPERADO}' en la carpeta actual o en Recursos/.")

HASH_ORIGINAL_INICIO = sha256_file(origen_path)
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"ETL_{RUN_TIMESTAMP}"
copia_raw_path = RAW_DIR / f"raw_{RUN_TIMESTAMP}_{origen_path.name}"
shutil.copy2(origen_path, copia_raw_path)
assert sha256_file(copia_raw_path) == HASH_ORIGINAL_INICIO, "Fallo de integridad en copia Raw."

wb = openpyxl.load_workbook(copia_raw_path, read_only=True, data_only=False)
HOJAS_EXCEL = wb.sheetnames
sheet_name = HOJAS_EXCEL[0]
ws = wb[sheet_name]
MAX_ROW, MAX_COL = ws.max_row, ws.max_column
wb.close()

df_raw = pd.read_excel(copia_raw_path, sheet_name=sheet_name, dtype=object)
print(f"Ejecución: {RUN_ID} | Hoja: {sheet_name} | Dimensiones: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
print(f"SHA-256 Original: {HASH_ORIGINAL_INICIO}")

In [ ]:
# ==============================================================================
# 2.2 TRANSFORMACIÓN ETL MEJORADA: NORMALIZACIÓN, REGEX/NLP Y UNPIVOT 2008-2026
# ==============================================================================
df_work = df_raw.copy()
assert df_work.shape[1] == 130, f"Se esperaban 130 columnas, se encontraron {df_work.shape[1]}"

COLS_2024_DET = [
    "2024_01_ene", "2024_02_feb", "2024_03_mar", "2024_03_bonos",
    "2024_04_abr", "2024_04_bonos", "2024_05_may", "2024_05_bonos",
    "2024_06_jun", "2024_06_bonos", "2024_07_jul", "2024_07_bonos",
    "2024_08_ago", "2024_08_bonos", "2024_09_sep", "2024_09_bonos",
    "2024_10_oct", "2024_10_bonos", "2024_11_nov", "2024_11_bonos",
    "2024_12_dic", "2024_12_bonos", "2024_banquete"
]

COLS_2025_DET = [
    "2025_01_ene", "2025_02_feb", "2025_03_mar", "2025_03_bonos",
    "2025_04_abr", "2025_04_bonos", "2025_05_may", "2025_05_bonos",
    "2025_06_jun", "2025_06_bonos", "2025_07_jul", "2025_07_bonos",
    "2025_08_ago", "2025_08_bonos", "2025_09_sep", "2025_09_bonos",
    "2025_10_oct", "2025_10_bonos", "2025_11_nov", "2025_11_bonos",
    "2025_12_dic", "2025_12_bonos"
]

COLS_2026_DET = [
    "2026_01_ene", "2026_02_feb", "2026_03_mar", "2026_03_bonos",
    "2026_04_abr", "2026_04_bonos", "2026_05_may", "2026_05_bonos",
    "2026_06_jun", "2026_06_bonos", "2026_07_jul", "2026_07_bonos",
    "2026_08_ago", "2026_08_bonos", "2026_09_sep", "2026_09_bonos",
    "2026_10_oct", "2026_10_bonos", "2026_11_nov", "2026_11_bonos",
    "2026_12_dic"
]

NOMBRES_LIMPIOS = (
    [
        "cod", "fecha_ingreso_raw", "socio_raw", "trato_raw", "nombre_completo_token",
        "nombres_token", "apellidos_token", "cedula_token", "telefono_token",
        "direccion_token", "barrio_token", "municipio_raw", "dia_cump_raw",
        "mes_cump_raw", "anio_nac_raw", "lo_atendera_raw", "mensajes_cumpleanos",
        "detalle_cumpleanos", "whatsapp_flag", "veces_en_anio_raw",
        "trim_2026_ene_mar", "trim_2026_abr_jun", "trim_2026_jul_sep", "trim_2026_oct_dic",
        "encuesta_eventos", "email_token", "certificado_donacion_raw", "donacion_extra_raw",
        "gestion_log", "llamar_bonos_2026", "calamidad", "enfermo",
        "observaciones_comentarios", "tipo_pago_raw", "recogerle_dia", "frecuencia_raw",
        "recibos_preferencia_raw", "alcancia_raw", "bono_2026_raw", "cantarilla_2026_raw",
        "rifa_virgen_carmen_raw", "cuota_raw", "ult_pago_raw", "recibos_numeros"
    ]
    + [f"total_{y}" for y in range(2008, 2024)]
    + COLS_2024_DET
    + ["total_2024"]
    + COLS_2025_DET
    + ["total_2025"]
    + COLS_2026_DET
    + ["total_2026", "acumulado_reportado"]
)
df_work.columns = NOMBRES_LIMPIOS
df_work["id_benefactor"] = range(1, len(df_work) + 1)
df_work["cod"] = pd.to_numeric(df_work["cod"], errors="coerce").astype("Int64")

def clean_str(val):
    if pd.isna(val):
        return np.nan
    s = re.sub(r"\s+", " ", str(val)).strip()
    return s if s != "" else np.nan

TEXT_COLS = [
    "socio_raw", "trato_raw", "nombre_completo_token", "nombres_token", "apellidos_token",
    "cedula_token", "telefono_token", "direccion_token", "barrio_token", "municipio_raw",
    "mes_cump_raw", "lo_atendera_raw", "mensajes_cumpleanos", "detalle_cumpleanos",
    "whatsapp_flag", "veces_en_anio_raw", "encuesta_eventos", "email_token",
    "certificado_donacion_raw", "donacion_extra_raw", "gestion_log", "llamar_bonos_2026",
    "calamidad", "enfermo", "observaciones_comentarios", "tipo_pago_raw", "frecuencia_raw",
    "recibos_preferencia_raw", "alcancia_raw", "bono_2026_raw", "cantarilla_2026_raw",
    "rifa_virgen_carmen_raw"
]
for c in TEXT_COLS:
    df_work[c] = df_work[c].apply(clean_str)

# Homologación de SOCIO, TIPO PAGO, MUNICIPIO, TRATO y TELEMERCADERISTA
def normalizar_socio(val):
    if pd.isna(val):
        return "SIN_CLASIFICAR"
    v = val.upper()
    if "RETIR" in v or "NO SIGUE" in v or "NO SEG" in v:
        return "RETIRADO"
    if "PARROQUIA" in v:
        return "VISITA_PARROQUIAS"
    if "BONO" in v:
        return "BONOS"
    if "PERM" in v:
        return "PERMANENTE"
    if "ESP" in v:
        return "ESPORADICO"
    if "BANQUETE" in v:
        return "BANQUETE"
    if "CANTARILLA" in v or "RIFA" in v:
        return "RIFA_CANTARILLA"
    if "ALC" in v:
        return "ALCANCIA"
    if "BINGO" in v:
        return "BINGO"
    if "NSE" in v or "TARDES" in v or "OYENTE" in v:
        return "EVENTO_NSE"
    if "EVENTO" in v or "PEREGRIN" in v or "RETIRO" in v or "EJERCICIOS" in v:
        return "EVENTO"
    return "OTRO"

df_work["socio_categoria"] = df_work["socio_raw"].apply(normalizar_socio)

VALID_TIPO_PAGO = {"DO": "DOMICILIO", "CO": "CONSIGNACION", "TR": "TRANSFERENCIA", "SE": "SEDE", "DE": "DEBITO"}
df_work["tipo_pago_norm"] = df_work["tipo_pago_raw"].map(
    lambda x: VALID_TIPO_PAGO.get(str(x).upper(), "OTRO_O_INVALIDO") if pd.notna(x) else "NO_ESPECIFICADO"
)

MESES_VALIDOS = {"ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"}
df_work["mes_cump_norm"] = df_work["mes_cump_raw"].map(
    lambda x: x.upper() if pd.notna(x) and x.upper() in MESES_VALIDOS else np.nan
)

def parse_excel_or_text_date(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, (int, float, np.integer, np.floating)):
        if 20000 <= float(val) <= 60000:
            return pd.to_datetime(float(val), unit="D", origin="1899-12-30", errors="coerce")
        return pd.NaT
    if isinstance(val, (datetime, pd.Timestamp)):
        return pd.to_datetime(val)
    s = str(val).strip().replace("//", "/").replace("'", "")
    return pd.to_datetime(s, dayfirst=True, errors="coerce")

df_work["fecha_ingreso_norm"] = df_work["fecha_ingreso_raw"].apply(parse_excel_or_text_date)
df_work["ult_pago_norm"] = df_work["ult_pago_raw"].apply(parse_excel_or_text_date)
df_work["flag_fecha_ingreso_invalida"] = df_work["fecha_ingreso_raw"].notna() & df_work["fecha_ingreso_norm"].isna()
df_work["flag_ult_pago_invalido"] = df_work["ult_pago_raw"].notna() & df_work["ult_pago_norm"].isna()

# Conversión numérica de montos y recaudo
df_work["cuota_valor_num"] = pd.to_numeric(df_work["cuota_raw"], errors="coerce")
df_work["bono_2026_valor_num"] = pd.to_numeric(df_work["bono_2026_raw"], errors="coerce")
df_work["recogerle_dia_num"] = pd.to_numeric(df_work["recogerle_dia"], errors="coerce").astype("Int64")

# Procesamiento NLP/Regex sobre comentarios y campos descriptivos
REGEX_FAMILIA = re.compile(
    r"\b(esposo|esposa|esposos|familia|flia|mam[aá]|pap[aá]|hijo|hija|herman[oa]s?|sobrin[oa]s?|t[ií][oa]s?|amig[oa]s?|compa[ñn][ií]a|junto con|dona con)\b",
    re.IGNORECASE
)
REGEX_NO_CONTACTO = re.compile(
    r"\b(no llamar|no contactar|retirar|no sigue|no seg|falleci[oó]|muri[oó]|no felicitar|no molestar)\b",
    re.IGNORECASE
)

COLS_TEXTO_INTEGRADO = [
    "socio_raw", "trato_raw", "mensajes_cumpleanos", "detalle_cumpleanos",
    "encuesta_eventos", "donacion_extra_raw", "gestion_log", "llamar_bonos_2026",
    "calamidad", "enfermo", "observaciones_comentarios", "recibos_preferencia_raw",
    "alcancia_raw", "bono_2026_raw", "cantarilla_2026_raw", "rifa_virgen_carmen_raw"
]

def extraer_texto_consolidado(row):
    partes = [str(row[c]) for c in COLS_TEXTO_INTEGRADO if pd.notna(row[c])]
    return " | ".join(partes)

df_work["texto_consolidado_crm"] = df_work.apply(extraer_texto_consolidado, axis=1)

def detectar_vinculo_familiar(row):
    txt = row["texto_consolidado_crm"]
    trato = str(row["trato_raw"]) if pd.notna(row["trato_raw"]) else ""
    matches = REGEX_FAMILIA.findall(txt)
    if not matches and re.search(r"(esposos|se[ñn]ores|familia|flia)", trato, re.IGNORECASE):
        matches = ["pareja/familia"]
    if matches:
        terminos = sorted(set(m.lower() for m in matches))
        return True, ", ".join(terminos)
    return False, "No documentado"

fam_res = df_work.apply(detectar_vinculo_familiar, axis=1)
df_work["flag_aporte_familiar_amigos"] = [r[0] for r in fam_res]
df_work["detalle_vinculo_familiar"] = [r[1] for r in fam_res]

def detectar_restriccion(row):
    txt = row["texto_consolidado_crm"]
    matches = REGEX_NO_CONTACTO.findall(txt)
    if matches:
        return True, ", ".join(sorted(set(m.upper() for m in matches)))
    return False, "NINGUNA"

rest_res = df_work.apply(detectar_restriccion, axis=1)
df_work["flag_restriccion_no_contacto"] = [r[0] for r in rest_res]
df_work["motivo_restriccion_contacto"] = [r[1] for r in rest_res]

# Canal preferido y bandera de datos sensibles pastorales/salud
df_work["flag_whatsapp"] = (
    df_work["whatsapp_flag"].fillna("").str.upper().str.contains("SI")
    | df_work["mensajes_cumpleanos"].fillna("").str.upper().str.contains("WHATSAPP")
)
df_work["canal_preferido"] = np.where(df_work["flag_whatsapp"], "WhatsApp / Telefónico", "Llamada Telefónica")
df_work["flag_nota_sensible_pastoral"] = df_work["enfermo"].notna() | df_work["calamidad"].notna()

# Duplicados de cédula y teléfonos compartidos
tel_freq = df_work["telefono_token"].value_counts(dropna=True)
df_work["personas_en_telefono"] = df_work["telefono_token"].map(tel_freq).fillna(0).astype(int)
df_work["flag_telefono_compartido"] = df_work["personas_en_telefono"] > 1

ced_freq = df_work["cedula_token"].value_counts(dropna=True)
df_work["flag_cedula_duplicada"] = df_work["cedula_token"].map(ced_freq).fillna(0).astype(int) > 1

# Unpivot (Melt) de las columnas financieras 2008-2026
cols_anuales_hist = [f"total_{y}" for y in range(2008, 2024)]
cols_mensuales_detalle = COLS_2024_DET + COLS_2025_DET + COLS_2026_DET

for c in cols_anuales_hist + cols_mensuales_detalle + ["total_2024", "total_2025", "total_2026", "acumulado_reportado"]:
    df_work[c] = pd.to_numeric(df_work[c], errors="coerce")

df_hist_long = df_work.melt(
    id_vars=["id_benefactor", "lo_atendera_raw"],
    value_vars=cols_anuales_hist,
    var_name="origen_col",
    value_name="monto"
).dropna(subset=["monto"])
df_hist_long = df_hist_long[df_hist_long["monto"] > 0].copy()
df_hist_long["anio"] = df_hist_long["origen_col"].str.extract(r"(\d{4})").astype(int)
df_hist_long["mes"] = 12
df_hist_long["recurso"] = "Donación directa"
df_hist_long["tipo_registro"] = "ANUAL_HISTORICO"

df_mensual_long = df_work.melt(
    id_vars=["id_benefactor", "lo_atendera_raw"],
    value_vars=cols_mensuales_detalle,
    var_name="origen_col",
    value_name="monto"
).dropna(subset=["monto"])
df_mensual_long = df_mensual_long[df_mensual_long["monto"] > 0].copy()
df_mensual_long["anio"] = df_mensual_long["origen_col"].str.slice(0, 4).astype(int)
df_mensual_long["mes"] = df_mensual_long["origen_col"].str.extract(r"^\d{4}_(\d{2})_")[0].fillna(12).astype(int)

def mapear_recurso_oficial(col_name: str) -> str:
    if "bonos" in col_name:
        return "Bono"
    if "banquete" in col_name:
        return "Banquete navideño"
    return "Donación directa"

df_mensual_long["recurso"] = df_mensual_long["origen_col"].apply(mapear_recurso_oficial)
df_mensual_long["tipo_registro"] = np.where(
    df_mensual_long["recurso"] == "Bono", "MENSUAL_BONO",
    np.where(df_mensual_long["recurso"] == "Banquete navideño", "BANQUETE", "MENSUAL_REGULAR")
)

df_aportes_long = pd.concat([df_hist_long, df_mensual_long], ignore_index=True)
df_aportes_long["fecha_referencia"] = pd.to_datetime(
    df_aportes_long["anio"].astype(str) + "-" + df_aportes_long["mes"].astype(str).str.zfill(2) + "-15"
)
df_aportes_long = df_aportes_long.sort_values(["id_benefactor", "fecha_referencia"]).reset_index(drop=True)
df_aportes_long["id_aporte"] = range(1, len(df_aportes_long) + 1)

print(f"ETL completado:")
print(f" - Total benefactores procesados          : {len(df_work):,}")
print(f" - Total transacciones confirmadas (> $0) : {len(df_aportes_long):,}")
print(f" - Valor acumulado total verificado       : COP ${df_aportes_long['monto'].sum():,.0f}")
print(f" - Benefactores con vínculo familiar/amigo: {int(df_work['flag_aporte_familiar_amigos'].sum()):,}")
print(f" - Benefactores con restricción contacto  : {int(df_work['flag_restriccion_no_contacto'].sum()):,}")
print(f" - Benefactores en teléfonos compartidos  : {int(df_work['flag_telefono_compartido'].sum()):,} ({int(df_work.loc[df_work['flag_telefono_compartido'], 'telefono_token'].nunique())} grupos)")

## 3. Módulo 3 – Clasificación y Comportamiento Histórico de los Benefactores
Calcula las métricas individuales de cada benefactor (**fecha del último aporte, número de aportes, intervalo promedio en meses, valor acumulado, valor promedio, recurso principal, comportamiento histórico y vínculo familiar**) y aplica las reglas analíticas objetivas para clasificar el 100 % del padrón en las 4 categorías exigidas:
- **Permanente o recurrente**
- **Esporádico**
- **Único aporte registrado**
- **Sin aportes recientes que requieren revisión**

In [ ]:
# ==============================================================================
# 3.1 CÁLCULO DE MÉTRICAS RFM, FRECUENCIA, ÚLTIMO APORTE Y CLASIFICACIÓN
# ==============================================================================
# Agregaciones por benefactor desde fact_aportes
agg_total = df_aportes_long.groupby("id_benefactor").agg(
    num_aportes_registrados=("id_aporte", "count"),
    valor_acumulado=("monto", "sum"),
    valor_promedio_aporte=("monto", "mean"),
    fecha_primer_aporte_tx=("fecha_referencia", "min"),
    fecha_ultimo_aporte_tx=("fecha_referencia", "max"),
    primer_anio_aporte=("anio", "min"),
    ultimo_anio_aporte=("anio", "max"),
    anios_distintos_activos=("anio", "nunique")
).reset_index()

# Aportes en ventana reciente (2024-2026) vs histórica previa (2008-2024)
agg_reciente = (
    df_aportes_long[df_aportes_long["anio"] >= 2024]
    .groupby("id_benefactor")
    .agg(
        aportes_recientes_2024_2026=("id_aporte", "count"),
        monto_reciente_2024_2026=("monto", "sum")
    )
    .reset_index()
)

# Ventana de entrenamiento para IA sin data leakage (2008-2024) y ventana objetivo (2025-2026)
agg_pre2025 = (
    df_aportes_long[df_aportes_long["anio"] <= 2024]
    .groupby("id_benefactor")
    .agg(
        num_aportes_2008_2024=("id_aporte", "count"),
        monto_acum_2008_2024=("monto", "sum"),
        monto_prom_2008_2024=("monto", "mean"),
        anios_activos_2008_2024=("anio", "nunique")
    )
    .reset_index()
)

agg_2024 = (
    df_aportes_long[df_aportes_long["anio"] == 2024]
    .groupby("id_benefactor")
    .agg(aportes_2024_count=("id_aporte", "count"), monto_2024_sum=("monto", "sum"))
    .reset_index()
)

agg_2025_2026 = (
    df_aportes_long[df_aportes_long["anio"] >= 2025]
    .groupby("id_benefactor")
    .agg(aportes_2025_2026_count=("id_aporte", "count"), monto_2025_2026_sum=("monto", "sum"))
    .reset_index()
)

# Intervalo promedio en meses entre aportes consecutivos
def calcular_intervalos_meses(df_tx: pd.DataFrame) -> pd.Series:
    df_sorted = df_tx.sort_values(["id_benefactor", "fecha_referencia"])
    diffs = df_sorted.groupby("id_benefactor")["fecha_referencia"].apply(
        lambda s: ((s.max() - s.min()).days / 30.4375) / max(len(s) - 1, 1) if len(s) >= 2 else np.nan
    )
    return diffs.rename("intervalo_promedio_meses")

s_intervalos = calcular_intervalos_meses(df_aportes_long).reset_index()

# Recurso al que más contribuye cada benefactor (por monto y frecuencia)
recurso_top_tx = (
    df_aportes_long.groupby(["id_benefactor", "recurso"])
    .agg(monto_recurso=("monto", "sum"), tx_recurso=("id_aporte", "count"))
    .reset_index()
    .sort_values(["id_benefactor", "monto_recurso", "tx_recurso"], ascending=[True, False, False])
    .drop_duplicates("id_benefactor")
    .rename(columns={"recurso": "recurso_top_transaccional", "monto_recurso": "monto_recurso_top"})
)

# Unir métricas al padrón maestro
df_master = (
    df_work
    .merge(agg_total, on="id_benefactor", how="left")
    .merge(agg_reciente, on="id_benefactor", how="left")
    .merge(agg_pre2025, on="id_benefactor", how="left")
    .merge(agg_2024, on="id_benefactor", how="left")
    .merge(agg_2025_2026, on="id_benefactor", how="left")
    .merge(s_intervalos, on="id_benefactor", how="left")
    .merge(recurso_top_tx[["id_benefactor", "recurso_top_transaccional", "monto_recurso_top"]], on="id_benefactor", how="left")
)

for col_zero in [
    "num_aportes_registrados", "valor_acumulado", "valor_promedio_aporte",
    "aportes_recientes_2024_2026", "monto_reciente_2024_2026",
    "num_aportes_2008_2024", "monto_acum_2008_2024", "monto_prom_2008_2024",
    "anios_activos_2008_2024", "aportes_2024_count", "monto_2024_sum",
    "aportes_2025_2026_count", "monto_2025_2026_sum", "anios_distintos_activos",
    "primer_anio_aporte", "ultimo_anio_aporte"
]:
    df_master[col_zero] = df_master[col_zero].fillna(0)

# Fecha del último aporte consolidada (máximo entre ult_pago_norm y fecha_ultimo_aporte_tx)
df_master["fecha_ultimo_aporte"] = df_master[["ult_pago_norm", "fecha_ultimo_aporte_tx"]].max(axis=1)

# Determinar recurso principal combinando evidencia transaccional y categoría de socio/boletas
MAPA_SOCIO_RECURSO = {
    "BONOS": "Bono",
    "PERMANENTE": "Donación directa",
    "ESPORADICO": "Donación directa",
    "BANQUETE": "Banquete navideño",
    "RIFA_CANTARILLA": "Rifa cantarrilla",
    "EVENTO": "Evento",
    "EVENTO_NSE": "Evento",
    "BINGO": "Bingo",
    "VISITA_PARROQUIAS": "Visita a parroquias",
    "ALCANCIA": "Otro recurso configurable"
}

def determinar_recurso_principal(row):
    if pd.notna(row["recurso_top_transaccional"]):
        # Si no tiene bono ni banquete como top, pero su vínculo principal es un recurso específico (ej. Evento, Rifa, Bono)
        if row["recurso_top_transaccional"] == "Donación directa" and row["socio_categoria"] in ["BONOS", "BANQUETE", "RIFA_CANTARILLA", "BINGO", "VISITA_PARROQUIAS"]:
            if row["valor_acumulado"] == 0:
                return MAPA_SOCIO_RECURSO[row["socio_categoria"]]
        return row["recurso_top_transaccional"]
    return MAPA_SOCIO_RECURSO.get(row["socio_categoria"], "Sin registro de recurso")

df_master["recurso_mayor_contribucion"] = df_master.apply(determinar_recurso_principal, axis=1)

# Regla analítica de clasificación en las 4 categorías obligatorias
def clasificar_benefactor(row) -> str:
    n_tot = int(row["num_aportes_registrados"])
    n_rec = int(row["aportes_recientes_2024_2026"])
    if n_rec == 0:
        return "Sin aportes recientes que requieren revisión"
    elif n_tot == 1:
        return "Con un único aporte registrado"
    elif n_tot >= 4 and n_rec >= 2:
        return "Permanente o recurrente"
    else:
        return "Esporádico"

df_master["clasificacion_benefactor"] = df_master.apply(clasificar_benefactor, axis=1)

# Resumen de comportamiento histórico (sin inventar causas personales)
def describir_comportamiento(row) -> str:
    n_tot = int(row["num_aportes_registrados"])
    if n_tot == 0:
        return "Sin aportes monetarios confirmados en matriz 2008-2026; requiere validación de estado."
    m_2024 = float(row["monto_2024_sum"])
    m_25_26 = float(row["monto_2025_2026_sum"]) / 1.8  # anualizado aprox
    if row["aportes_recientes_2024_2026"] == 0:
        tendencia = f"Inactivo desde {int(row['ultimo_anio_aporte'])}"
    elif m_25_26 > m_2024 * 1.15:
        tendencia = "Tendencia creciente en 2025-2026"
    elif m_25_26 < m_2024 * 0.85 and m_2024 > 0:
        tendencia = "Tendencia decreciente frente a 2024"
    else:
        tendencia = "Aportación estable en período reciente"
    cadencia = (
        f"cada {row['intervalo_promedio_meses']:.1f} meses"
        if pd.notna(row["intervalo_promedio_meses"]) and row["intervalo_promedio_meses"] > 0
        else "aporte puntual"
    )
    return f"Activo {int(row['primer_anio_aporte'])}-{int(row['ultimo_anio_aporte'])} ({n_tot} aportes, {cadencia}); {tendencia}."

df_master["comportamiento_historico"] = df_master.apply(describir_comportamiento, axis=1)

print("=== DISTRIBUCIÓN OFICIAL DE CLASIFICACIÓN DE BENEFACTORES ===")
resumen_clasif = (
    df_master.groupby("clasificacion_benefactor")
    .agg(
        benefactores=("id_benefactor", "count"),
        aportes_promedio=("num_aportes_registrados", "mean"),
        intervalo_meses_promedio=("intervalo_promedio_meses", "mean"),
        valor_acumulado_total=("valor_acumulado", "sum"),
        valor_promedio_por_aporte=("valor_promedio_aporte", "mean"),
        con_vinculo_familiar=("flag_aporte_familiar_amigos", "sum")
    )
    .reset_index()
)
resumen_clasif["porcentaje_%"] = (resumen_clasif["benefactores"] / len(df_master) * 100).round(2)
display(resumen_clasif.sort_values("benefactores", ascending=False))

# Subdivisión analítica de 'Sin aportes recientes' y auditoría detallada de ULT PAGO (corte 2026-11-18)
inactivos_con_hist = int(((df_master["aportes_recientes_2024_2026"] == 0) & (df_master["num_aportes_registrados"] > 0)).sum())
sin_aportes_matriz = int((df_master["num_aportes_registrados"] == 0).sum())
print(f"\nDesglose de 'Sin aportes recientes ({inactivos_con_hist + sin_aportes_matriz})':")
print(f" - Con historial previo (2008-2023) pero sin aportes en 2024-2026: {inactivos_con_hist} benefactores")
print(f" - Sin aportes monetarios registrados en matriz (celda vacía)    : {sin_aportes_matriz} benefactores")

FECHA_CORTE = pd.Timestamp("2026-11-18")
FECHA_MIN_VALIDA = pd.Timestamp("2000-01-01")
ult_no_nulos = int(df_master["ult_pago_raw"].notna().sum())
ult_invalidos_txt = int(df_master["flag_ult_pago_invalido"].sum())
ult_pre_2000 = int((df_master["ult_pago_norm"] < FECHA_MIN_VALIDA).sum())
ult_post_corte = int((df_master["ult_pago_norm"] > FECHA_CORTE).sum())
mask_ult_valida = (df_master["ult_pago_norm"] >= FECHA_MIN_VALIDA) & (df_master["ult_pago_norm"] <= FECHA_CORTE)
ult_validas = df_master.loc[mask_ult_valida, "ult_pago_norm"]
dias_antiguedad = (FECHA_CORTE - ult_validas).dt.days

recencia_rangos = pd.DataFrame([
    ("0-30 días", int((dias_antiguedad <= 30).sum())),
    ("31-90 días", int(((dias_antiguedad >= 31) & (dias_antiguedad <= 90)).sum())),
    ("91-180 días", int(((dias_antiguedad >= 91) & (dias_antiguedad <= 180)).sum())),
    ("181-365 días", int(((dias_antiguedad >= 181) & (dias_antiguedad <= 365)).sum())),
    ("Más de 365 días", int((dias_antiguedad > 365).sum())),
], columns=["Rango de Antigüedad (ULT PAGO)", "Benefactores"])

print(f"\n=== AUDITORÍA Y RECENCIA DE 'ULT PAGO' (Corte: {FECHA_CORTE.strftime('%Y-%m-%d')}) ===")
print(f"No vacíos: {ult_no_nulos:,} | Fechas válidas en rango: {len(ult_validas):,} | Textos sin año: {ult_invalidos_txt} | < Año 2000: {ult_pre_2000} | > Fecha corte: {ult_post_corte}")
display(recencia_rangos)

In [ ]:
# ==============================================================================
# 3.2 FUNCIÓN DE CONSULTA DE FICHA INDIVIDUAL DE BENEFACTOR (PARA SUSTENTACIÓN)
# ==============================================================================
def consultar_ficha_benefactor(cod_o_id: int):
    filtro = df_master[(df_master["cod"] == cod_o_id) | (df_master["id_benefactor"] == cod_o_id)]
    if filtro.empty:
        print(f"No se encontró benefactor con COD o ID = {cod_o_id}")
        return
    b = filtro.iloc[0]
    ficha = pd.DataFrame([
        ("ID Benefactor / COD", f"{b['id_benefactor']} (COD={b['cod']})"),
        ("Token Nombre / Trato", f"{b['trato_raw']} {b['nombre_completo_token']}"),
        ("Telemercaderista Asignada", b["lo_atendera_raw"]),
        ("Clasificación Analítica", b["clasificacion_benefactor"]),
        ("1. Fecha del último aporte", str(b["fecha_ultimo_aporte"])[:10] if pd.notna(b["fecha_ultimo_aporte"]) else "Sin registro"),
        ("2. Número de aportes registrados", f"{int(b['num_aportes_registrados'])} aportes"),
        ("3. Frecuencia / Intervalo promedio", f"{b['intervalo_promedio_meses']:.1f} meses (Declarada: {b['frecuencia_raw']})" if pd.notna(b["intervalo_promedio_meses"]) else f"Único/Sin intervalo (Declarada: {b['frecuencia_raw']})"),
        ("4. Valor acumulado", f"COP ${b['valor_acumulado']:,.0f}"),
        ("5. Valor promedio de los aportes", f"COP ${b['valor_promedio_aporte']:,.0f}"),
        ("6. Recurso al que más contribuye", b["recurso_mayor_contribucion"]),
        ("7. Comportamiento histórico", b["comportamiento_historico"]),
        ("8. Aporta con familiares o amigos", f"{'SÍ' if b['flag_aporte_familiar_amigos'] else 'No documentado'} ({b['detalle_vinculo_familiar']})"),
        ("Teléfono compartido", f"{'SÍ' if b['flag_telefono_compartido'] else 'NO'} ({b['personas_en_telefono']} personas en {b['telefono_token']})"),
        ("Restricción de contacto", f"{'SÍ: ' + b['motivo_restriccion_contacto'] if b['flag_restriccion_no_contacto'] else 'NINGUNA'}")
    ], columns=["Atributo Exigido (Módulo 3)", "Valor Calculado"])
    display(ficha)

print("Ejemplo 1: Ficha de un benefactor permanente/recurrente (COD=3):")
consultar_ficha_benefactor(3)

print("\nEjemplo 2: Muestra de 8 benefactores con aportes documentados en compañía de familiares o amigos:")
cols_fam = ["cod", "nombre_completo_token", "socio_raw", "clasificacion_benefactor", "detalle_vinculo_familiar", "valor_acumulado"]
display(df_master[df_master["flag_aporte_familiar_amigos"]][cols_fam].head(8))

## 4. Módulo 4 – Inteligencia Artificial, Predicción de Afinidad por Recurso y Evaluación
Implementa el motor de afinidad para los **9 recursos oficiales de captación (`a` hasta `i`)**:
- Para los recursos con etiquetas históricas amplias (**Donación directa, Bono, Evento, Banquete navideño, Rifa cantarrilla**), entrena y compara clasificadores supervisados (**`RandomForestClassifier`** vs **`LogisticRegression`**) con validación cruzada estratificada (`5-Fold CV`) y conjunto de prueba (`20 %`).
- Para evitar fuga de datos (*data leakage*) en los dos recursos transaccionales continuos (**Donación directa** y **Bono**), el modelo supervisado utiliza variables de la ventana histórica previa (`2008–2024`) para predecir la participación efectiva en `2025–2026`.
- Para los recursos de nicho o baja frecuencia histórica (**Bingo, Visita a parroquias, Día de madres y padres, Otro recurso configurable**), aplica un **scoring híbrido RFM + similitud de eventos/texto** y explica explícitamente la diferencia entre un modelo predictivo supervisado y una segmentación basada en reglas.

In [ ]:
# ==============================================================================
# 4.1 CONSTRUCCIÓN DE ETIQUETAS POR RECURSO (CATÁLOGO a-i) Y VARIABLES PREDICTORAS
# ==============================================================================
CATALOGO_RECURSOS = [
    "a. Donación directa",
    "b. Bono",
    "c. Rifa cantarrilla",
    "d. Evento",
    "e. Bingo",
    "f. Visita a parroquias",
    "g. Banquete navideño",
    "h. Día de madres y padres",
    "i. Otro recurso configurable"
]

# Participación monetaria en 2025-2026 por recurso (para evaluación temporal limpia sin leakage)
tx_25_26_don = set(df_aportes_long[(df_aportes_long["anio"] >= 2025) & (df_aportes_long["recurso"] == "Donación directa")]["id_benefactor"])
tx_25_26_bono = set(df_aportes_long[(df_aportes_long["anio"] >= 2025) & (df_aportes_long["recurso"] == "Bono")]["id_benefactor"])
tx_all_don = set(df_aportes_long[df_aportes_long["recurso"] == "Donación directa"]["id_benefactor"])
tx_all_bono = set(df_aportes_long[df_aportes_long["recurso"] == "Bono"]["id_benefactor"])
tx_all_banq = set(df_aportes_long[df_aportes_long["recurso"] == "Banquete navideño"]["id_benefactor"])

socio_up = df_master["socio_raw"].fillna("").str.upper()
txt_up = df_master["texto_consolidado_crm"].fillna("").str.upper()

# Etiquetas históricas de evidencia real por cada uno de los 9 recursos
df_master["y_donacion_directa_25_26"] = df_master["id_benefactor"].isin(tx_25_26_don).astype(int)
df_master["y_bono_25_26"] = (
    df_master["id_benefactor"].isin(tx_25_26_bono)
    | df_master["bono_2026_raw"].notna()
    | df_master["llamar_bonos_2026"].notna()
).astype(int)

df_master["ev_donacion_directa"] = (
    df_master["id_benefactor"].isin(tx_all_don)
    | socio_up.str.contains("PERM|ESP")
    | df_master["donacion_extra_raw"].notna()
).astype(int)

df_master["ev_bono"] = (
    df_master["id_benefactor"].isin(tx_all_bono)
    | socio_up.str.contains("BONO")
    | df_master["bono_2026_raw"].notna()
    | df_master["llamar_bonos_2026"].notna()
).astype(int)

df_master["ev_rifa_cantarrilla"] = (
    df_master["cantarilla_2026_raw"].notna()
    | df_master["rifa_virgen_carmen_raw"].notna()
    | socio_up.str.contains("CANTARILLA|RIFA")
    | txt_up.str.contains("CANTARILLA|RIFA")
).astype(int)

df_master["ev_evento"] = (
    socio_up.str.contains("EVENTO|TARDES|NSE|PEREGRIN|RETIRO|EJERCICIOS|ANIVERSARIO|ORDEN")
    | df_master["encuesta_eventos"].notna()
    | txt_up.str.contains("EVENTO|PEREGRIN|RETIRO|TARDES")
).astype(int)

df_master["ev_bingo"] = (socio_up.str.contains("BINGO") | txt_up.str.contains(r"\bBINGO\b")).astype(int)
df_master["ev_visita_parroquias"] = (socio_up.str.contains("PARROQUIA") | txt_up.str.contains("PARROQUIA")).astype(int)
df_master["ev_banquete_navideno"] = (
    df_master["id_benefactor"].isin(tx_all_banq)
    | socio_up.str.contains("BANQUETE")
    | txt_up.str.contains("BANQUETE")
).astype(int)
df_master["ev_dia_madres_padres"] = txt_up.str.contains(r"\b(?:MADRE|PADRE|DIA DE LA MADRE|DIA DEL PADRE)\b", regex=True).astype(int)
df_master["ev_otro_recurso"] = (
    df_master["alcancia_raw"].notna()
    | socio_up.str.contains("ALC|CD|SOBRE|RADIOTON|TOLDOS")
).astype(int)

resumen_recursos = pd.DataFrame([
    ("a. Donación directa", int(df_master["ev_donacion_directa"].sum()), int(df_master["y_donacion_directa_25_26"].sum()), "Modelo Supervisado ML (Random Forest / LogReg)"),
    ("b. Bono", int(df_master["ev_bono"].sum()), int(df_master["y_bono_25_26"].sum()), "Modelo Supervisado ML (Random Forest / LogReg)"),
    ("c. Rifa cantarrilla", int(df_master["ev_rifa_cantarrilla"].sum()), int(df_master["ev_rifa_cantarrilla"].sum()), "Modelo Supervisado ML + Evidencia Boletas"),
    ("d. Evento", int(df_master["ev_evento"].sum()), int(df_master["ev_evento"].sum()), "Modelo Supervisado ML + Encuesta Eventos"),
    ("e. Bingo", int(df_master["ev_bingo"].sum()), int(df_master["ev_bingo"].sum()), "Motor Híbrido (Reglas RFM + Similitud Eventos)"),
    ("f. Visita a parroquias", int(df_master["ev_visita_parroquias"].sum()), int(df_master["ev_visita_parroquias"].sum()), "Motor Híbrido (Reglas RFM + Vínculo Parroquial)"),
    ("g. Banquete navideño", int(df_master["ev_banquete_navideno"].sum()), int(df_master["ev_banquete_navideno"].sum()), "Modelo Supervisado ML + Histórico Banquete"),
    ("h. Día de madres y padres", int(df_master["ev_dia_madres_padres"].sum()), int(df_master["ev_dia_madres_padres"].sum()), "Motor Híbrido (Reglas RFM + Vínculo Familiar)"),
    ("i. Otro recurso configurable", int(df_master["ev_otro_recurso"].sum()), int(df_master["ev_otro_recurso"].sum()), "Motor Híbrido (Reglas RFM + Alcancía/Especiales)")
], columns=["Recurso de Captación", "Benefactores con Evidencia Histórica", "Etiquetas Positivas Entrenamiento", "Estrategia Analítica Seleccionada"])
display(resumen_recursos)

In [ ]:
# ==============================================================================
# 4.2 ENTRENAMIENTO Y EVALUACIÓN DE MODELOS SUPERVISADOS (SCIKIT-LEARN)
# ==============================================================================
# Variables predictoras X basadas estrictamente en atributos históricos pre-2025 y demografía
antig_raw = np.where(
    df_master["primer_anio_aporte"] > 0,
    2026 - df_master["primer_anio_aporte"],
    np.where(df_master["fecha_ingreso_norm"].notna(), 2026 - df_master["fecha_ingreso_norm"].dt.year.fillna(2026), 0)
)
df_master["antiguedad_anios"] = np.clip(antig_raw, 0, 25)

def agrupar_municipio(m):
    if pd.isna(m):
        return "NO_REGISTRADO"
    mu = str(m).upper()
    if "MEDELLIN" in mu or "MEDELLÍN" in mu:
        return "MEDELLIN"
    if any(k in mu for k in ["ENVIGADO", "BELLO", "ITAGUI", "SABANETA", "COPACABANA", "ESTRELLA", "CALDAS"]):
        return "AREA_METROPOLITANA"
    return "OTRO_O_EXTERIOR"

def agrupar_trato(t):
    if pd.isna(t):
        return "NO_REGISTRADO"
    tu = str(t).upper()
    if "SEÑORA" in tu or "SENORA" in tu or "SEÑORITA" in tu:
        return "SENORA"
    if "SEÑOR" in tu or "SENOR" in tu:
        return "SENOR"
    return "OTRO_O_FAMILIA"

def agrupar_telemercaderista(r):
    if pd.isna(r):
        return "SIN_ASIGNAR"
    ru = str(r).upper()
    for principal in ["SANDRA", "VIVIANA", "LINA", "JORGE", "VICMAR", "JD", "NSE", "GLORIA"]:
        if principal in ru:
            return principal
    return "OTROS"

df_master["municipio_grupo"] = df_master["municipio_raw"].apply(agrupar_municipio)
df_master["trato_grupo"] = df_master["trato_raw"].apply(agrupar_trato)
df_master["telemercaderista_grupo"] = df_master["lo_atendera_raw"].apply(agrupar_telemercaderista)
df_master["flag_domicilio_recogerle"] = (
    (df_master["tipo_pago_norm"] == "DOMICILIO") | df_master["recogerle_dia_num"].notna()
).astype(int)

FEATURES_NUM = [
    "antiguedad_anios",
    "num_aportes_2008_2024",
    "monto_acum_2008_2024",
    "monto_prom_2008_2024",
    "anios_activos_2008_2024",
    "aportes_2024_count",
    "monto_2024_sum",
    "intervalo_promedio_meses",
    "personas_en_telefono",
    "flag_domicilio_recogerle"
]
FEATURES_BOOL = ["flag_aporte_familiar_amigos", "flag_whatsapp"]
FEATURES_CAT = ["municipio_grupo", "trato_grupo", "telemercaderista_grupo"]

X_all = df_master[FEATURES_NUM + FEATURES_BOOL + FEATURES_CAT].copy()
for bcol in FEATURES_BOOL:
    X_all[bcol] = X_all[bcol].astype(int)

def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), FEATURES_NUM + FEATURES_BOOL),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), FEATURES_CAT)
        ]
    )

TARGETS_ML = {
    "a. Donación directa": df_master["y_donacion_directa_25_26"],
    "b. Bono": df_master["y_bono_25_26"],
    "c. Rifa cantarrilla": df_master["ev_rifa_cantarrilla"],
    "d. Evento": df_master["ev_evento"],
    "g. Banquete navideño": df_master["ev_banquete_navideno"]
}

modelos_entrenados = {}
metricas_evaluacion = []
probabilidades_ml = {}

for nombre_rec, y_target in TARGETS_ML.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_target, test_size=0.20, stratify=y_target, random_state=42
    )
    rf_pipe = Pipeline([
        ("prep", build_preprocessor()),
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=5,
            class_weight="balanced", random_state=42, n_jobs=-1
        ))
    ])
    lr_pipe = Pipeline([
        ("prep", build_preprocessor()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ])

    rf_pipe.fit(X_train, y_train)
    lr_pipe.fit(X_train, y_train)

    y_pred_rf = rf_pipe.predict(X_test)
    y_prob_rf = rf_pipe.predict_proba(X_test)[:, 1]
    y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_auc_rf = cross_val_score(rf_pipe, X_all, y_target, cv=cv, scoring="roc_auc").mean()
    cv_auc_lr = cross_val_score(lr_pipe, X_all, y_target, cv=cv, scoring="roc_auc").mean()

    cm_test = confusion_matrix(y_test, y_pred_rf)
    tn, fp, fn, tp = cm_test.ravel() if cm_test.size == 4 else (0, 0, 0, 0)
    especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metricas_evaluacion.append({
        "Recurso": nombre_rec,
        "Algoritmo": "RandomForestClassifier (Seleccionado)",
        "Accuracy_Test": round(accuracy_score(y_test, y_pred_rf), 4),
        "Precision_Test": round(precision_score(y_test, y_pred_rf, zero_division=0), 4),
        "Recall_Test": round(recall_score(y_test, y_pred_rf, zero_division=0), 4),
        "Especificidad_Test": round(especificidad, 4),
        "F1_Score_Test": round(f1_score(y_test, y_pred_rf, zero_division=0), 4),
        "ROC_AUC_Test": round(roc_auc_score(y_test, y_prob_rf), 4),
        "Brier_Score_Test": round(brier_score_loss(y_test, y_prob_rf), 4),
        "ROC_AUC_5Fold_CV": round(cv_auc_rf, 4),
        "ROC_AUC_LogReg_Baseline": round(cv_auc_lr, 4)
    })

    # Re-entrenar o predecir probabilidades sobre todo el padrón para el recomendador
    modelos_entrenados[nombre_rec] = (rf_pipe, X_test, y_test, y_pred_rf, y_prob_rf)
    probabilidades_ml[nombre_rec] = rf_pipe.predict_proba(X_all)[:, 1]

df_metricas_ml = pd.DataFrame(metricas_evaluacion)
print("=== RESULTADOS DE EVALUACIÓN DE LOS MODELOS DE IA (TEST 20% + 5-FOLD CV) ===")
display(df_metricas_ml)

# Guardar probabilidades para los 9 recursos en df_master
df_master["prob_a_donacion_directa"] = probabilidades_ml["a. Donación directa"]
df_master["prob_b_bono"] = probabilidades_ml["b. Bono"]
df_master["prob_c_rifa_cantarrilla"] = np.maximum(probabilidades_ml["c. Rifa cantarrilla"], df_master["ev_rifa_cantarrilla"] * 0.92)
df_master["prob_d_evento"] = np.maximum(probabilidades_ml["d. Evento"], df_master["ev_evento"] * 0.92)
df_master["prob_g_banquete_navideno"] = np.maximum(probabilidades_ml["g. Banquete navideño"], df_master["ev_banquete_navideno"] * 0.92)

# Scoring híbrido normalizado [0, 1] para recursos de baja frecuencia histórica (e, f, h, i)
rfm_base = (
    0.40 * (df_master["num_aportes_registrados"].rank(pct=True))
    + 0.35 * (df_master["valor_acumulado"].rank(pct=True))
    + 0.25 * (df_master["aportes_recientes_2024_2026"] > 0).astype(float)
)
df_master["prob_e_bingo"] = np.clip(0.55 * df_master["ev_bingo"] + 0.25 * df_master["prob_c_rifa_cantarrilla"] + 0.20 * rfm_base, 0, 0.99)
df_master["prob_f_visita_parroquias"] = np.clip(0.55 * df_master["ev_visita_parroquias"] + 0.25 * df_master["prob_d_evento"] + 0.20 * rfm_base, 0, 0.99)
df_master["prob_h_dia_madres_padres"] = np.clip(0.50 * df_master["ev_dia_madres_padres"] + 0.30 * df_master["flag_aporte_familiar_amigos"].astype(float) + 0.20 * rfm_base, 0, 0.99)
df_master["prob_i_otro_recurso"] = np.clip(0.55 * df_master["ev_otro_recurso"] + 0.45 * rfm_base, 0, 0.99)

In [ ]:
# ==============================================================================
# 4.3 GRÁFICOS DE EVALUACIÓN DEL MODELO (MATRIZ DE CONFUSIÓN, ROC Y VARIABLES)
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Matriz de confusión para Donación Directa (2025-2026)
rf_don, X_t_don, y_t_don, y_p_don, y_pr_don = modelos_entrenados["a. Donación directa"]
cm = confusion_matrix(y_t_don, y_p_don)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Matriz de Confusión (Test 20%)\na. Donación directa (2025-2026)")
axes[0].set_xlabel("Predicción del Modelo")
axes[0].set_ylabel("Valor Real")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["No participa (0)", "Participa (1)"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["No participa (0)", "Participa (1)"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", color="black", fontsize=12, fontweight="bold")

# 2. Curvas ROC de los 5 modelos supervisados
for nombre_rec, (_, _, yt, _, ypr) in modelos_entrenados.items():
    fpr, tpr, _ = roc_curve(yt, ypr)
    auc_val = roc_auc_score(yt, ypr)
    axes[1].plot(fpr, tpr, lw=2, label=f"{nombre_rec} (AUC={auc_val:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_title("Curvas ROC en Conjunto de Prueba (Test 20%)")
axes[1].set_xlabel("Tasa de Falsos Positivos (FPR)")
axes[1].set_ylabel("Tasa de Verdaderos Positivos (TPR)")
axes[1].legend(loc="lower right", fontsize=8)
axes[1].grid(alpha=0.3)

# 3. Importancia de variables (Feature Importance) en Random Forest
ohe_names = rf_don.named_steps["prep"].named_transformers_["cat"].named_steps["ohe"].get_feature_names_out(FEATURES_CAT).tolist()
all_feat_names = FEATURES_NUM + FEATURES_BOOL + ohe_names
importances = pd.Series(rf_don.named_steps["clf"].feature_importances_, index=all_feat_names).sort_values(ascending=True).tail(10)
axes[2].barh(importances.index, importances.values, color="#015583")
axes[2].set_title("Top 10 Variables Más Importantes\n(Random Forest - Donación Directa)")
axes[2].set_xlabel("Importancia relativa (Gini)")
axes[2].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 4.4 PROTOTIPO DE RECOMENDACIÓN POR TELEMERCADERISTA, FECHA Y RECURSO (a-i)
# ==============================================================================
MAPA_COL_PROB = {
    "a. Donación directa": ("prob_a_donacion_directa", "ev_donacion_directa", "Modelo Supervisado ML (RandomForest)"),
    "b. Bono": ("prob_b_bono", "ev_bono", "Modelo Supervisado ML (RandomForest)"),
    "c. Rifa cantarrilla": ("prob_c_rifa_cantarrilla", "ev_rifa_cantarrilla", "Modelo Supervisado ML + Histórico Boletas"),
    "d. Evento": ("prob_d_evento", "ev_evento", "Modelo Supervisado ML + Encuesta Eventos"),
    "e. Bingo": ("prob_e_bingo", "ev_bingo", "Motor Híbrido (RFM + Similitud Eventos)"),
    "f. Visita a parroquias": ("prob_f_visita_parroquias", "ev_visita_parroquias", "Motor Híbrido (RFM + Vínculo Parroquial)"),
    "g. Banquete navideño": ("prob_g_banquete_navideno", "ev_banquete_navideno", "Modelo Supervisado ML + Histórico Banquete"),
    "h. Día de madres y padres": ("prob_h_dia_madres_padres", "ev_dia_madres_padres", "Motor Híbrido (RFM + Vínculo Familiar)"),
    "i. Otro recurso configurable": ("prob_i_otro_recurso", "ev_otro_recurso", "Motor Híbrido (RFM + Alcancías/Especiales)")
}

def recomendar_benefactores(
    telemercaderista: str = "Sandra",
    fecha_gestion: str = "2026-11-23",
    recurso_captacion: str = "b. Bono",
    top_n: int = 12
) -> pd.DataFrame:
    if recurso_captacion not in MAPA_COL_PROB:
        raise ValueError(f"Recurso no válido. Opciones: {list(MAPA_COL_PROB.keys())}")

    col_prob, col_ev, metodo = MAPA_COL_PROB[recurso_captacion]

    # Filtrar benefactores sin restricción de no contacto
    candidatos = df_master[~df_master["flag_restriccion_no_contacto"]].copy()

    if telemercaderista and telemercaderista.upper() != "TODOS":
        candidatos = candidatos[
            candidatos["lo_atendera_raw"].fillna("").str.upper().str.contains(telemercaderista.upper())
        ].copy()

    candidatos["afinidad_prob_%"] = (candidatos[col_prob] * 100).round(1)
    candidatos["participo_antes_recurso"] = np.where(candidatos[col_ev] == 1, "SÍ", "NO (Potencial por perfil)")
    candidatos["fecha_gestion_programada"] = fecha_gestion
    candidatos["motor_estimacion"] = metodo
    candidatos["alerta_telefono"] = np.where(
        candidatos["flag_telefono_compartido"],
        "TEL. COMPARTIDO (" + candidatos["personas_en_telefono"].astype(str) + " pers.) - Verificar agenda",
        "Teléfono individual"
    )

    cols_out = [
        "id_benefactor", "cod", "nombre_completo_token", "lo_atendera_raw",
        "telefono_token", "alerta_telefono", "clasificacion_benefactor",
        "participo_antes_recurso", "afinidad_prob_%", "num_aportes_registrados",
        "valor_acumulado", "fecha_ultimo_aporte", "canal_preferido"
    ]
    ranking = candidatos.sort_values(["afinidad_prob_%", "valor_acumulado"], ascending=[False, False]).head(top_n)
    print(f"=== RECOMENDACIÓN IA | Telemercaderista: {telemercaderista} | Fecha: {fecha_gestion} | Recurso: {recurso_captacion} ===")
    print(f"Motor utilizado: {metodo} | Candidatos elegibles (sin restricción): {len(candidatos)}")
    return ranking[cols_out]

display(recomendar_benefactores(telemercaderista="Sandra", fecha_gestion="2026-11-23", recurso_captacion="b. Bono", top_n=10))
display(recomendar_benefactores(telemercaderista="Viviana", fecha_gestion="2026-11-23", recurso_captacion="g. Banquete navideño", top_n=8))

## 5. Módulo 5 – Gestión de Telemercaderistas y Control de Teléfonos Compartidos
Implementa la funcionalidad completa para organizar las actividades de contacto (con **Viviana, Sandra, Hna Lina** y registro dinámico de otros usuarios), contemplando los 9 campos exigidos y el mecanismo anti-saturación que **conserva a cada benefactor como persona independiente** mientras coordina los contactos sobre números telefónicos compartidos.

### Caso Obligatorio de Prueba: Los 4 Hermanos (Juan, Felipe, Jesús y Daniel)
Se verifica explícitamente qué ocurre cuando 4 telemercaderistas distintas intentan contactar el mismo día a **Juan, Felipe, Jesús y Daniel**, quienes comparten el mismo teléfono pero tienen intereses y responsables diferentes.

In [ ]:
# ==============================================================================
# 5.1 MOTOR DE GESTIÓN DE CONTACTOS Y ANTI-SATURACIÓN DE TELÉFONOS COMPARTIDOS
# ==============================================================================
class GestorContactosTelemercadeo:
    def __init__(self, df_benefactores: pd.DataFrame, dias_enfriamiento_familiar: int = 2):
        self.df_padron = df_benefactores.copy()
        self.dias_enfriamiento = dias_enfriamiento_familiar
        self.telemercaderistas = sorted(
            set(["Viviana", "Sandra", "Hna Lina", "Jorge", "Vicmar", "JD", "NSE", "Gloria Trujillo"])
        )
        self.agenda = []

    def registrar_telemercaderista(self, nuevo_usuario: str):
        nombre = nuevo_usuario.strip()
        if nombre and nombre not in self.telemercaderistas:
            self.telemercaderistas.append(nombre)
            print(f"Usuario telemercaderista incorporado: '{nombre}'. Total activos: {len(self.telemercaderistas)}")

    def registrar_benefactor_en_padron(
        self, id_benefactor: int, cod: int, nombre: str, telefono: str,
        responsable: str, recurso_interes: str, horario_canal: str, restriccion: str = "NINGUNA"
    ):
        nuevo = {
            "id_benefactor": id_benefactor,
            "cod": cod,
            "nombre_completo_token": nombre,
            "telefono_token": telefono,
            "lo_atendera_raw": responsable,
            "recurso_mayor_contribucion": recurso_interes,
            "canal_preferido": horario_canal,
            "flag_restriccion_no_contacto": restriccion != "NINGUNA",
            "motivo_restriccion_contacto": restriccion,
            "fecha_ultimo_aporte": pd.Timestamp("2026-09-15")
        }
        self.df_padron = pd.concat([self.df_padron, pd.DataFrame([nuevo])], ignore_index=True)
        # Recalcular conteo de teléfonos compartidos sin fusionar personas
        freq = self.df_padron["telefono_token"].value_counts(dropna=True)
        self.df_padron["personas_en_telefono"] = self.df_padron["telefono_token"].map(freq).fillna(0).astype(int)
        self.df_padron["flag_telefono_compartido"] = self.df_padron["personas_en_telefono"] > 1

    def programar_contacto(
        self, id_benefactor: int, responsable: str, fecha_programada: str,
        recurso_gestionar: str, preferencias_horario_canal: str = None,
        auto_reprogramar_si_colision: bool = True
    ) -> dict:
        fila = self.df_padron[self.df_padron["id_benefactor"] == id_benefactor]
        if fila.empty:
            raise ValueError(f"No existe id_benefactor={id_benefactor}")
        b = fila.iloc[0]
        tel = str(b["telefono_token"])
        nombre = str(b["nombre_completo_token"])
        pref = preferencias_horario_canal or str(b.get("canal_preferido", "Llamada / Mañana"))
        ult_contacto = str(b.get("fecha_ultimo_aporte", "Sin contacto previo"))[:10]

        # 1. Validar restricciones de no contacto
        if bool(b.get("flag_restriccion_no_contacto", False)):
            reg = {
                "id_gestion": len(self.agenda) + 1,
                "responsable_gestion": responsable,
                "fecha_programada": fecha_programada,
                "recurso_que_se_desea_gestionar": recurso_gestionar,
                "id_benefactor": id_benefactor,
                "benefactor_asignado": nombre,
                "telefono_token": tel,
                "telefono_compartido": bool(b.get("flag_telefono_compartido", False)),
                "personas_en_telefono": int(b.get("personas_en_telefono", 1)),
                "ultimo_contacto_realizado": ult_contacto,
                "resultado_gestion": "BLOQUEADO_POR_RESTRICCION",
                "proxima_accion": "No contactar (Respetar solicitud de exclusión)",
                "preferencias_horario_canal": pref,
                "restricciones_contacto": str(b.get("motivo_restriccion_contacto", "NO_LLAMAR"))
            }
            self.agenda.append(reg)
            return reg

        # 2. Verificar si existen otros contactos pendientes o programados para el mismo teléfono
        contactos_mismo_tel = [
            g for g in self.agenda
            if g["telefono_token"] == tel and g["resultado_gestion"] in ["PROGRAMADO", "PENDIENTE"]
        ]
        fechas_ocupadas = {g["fecha_programada"]: g for g in contactos_mismo_tel}

        if fecha_programada not in fechas_ocupadas:
            estado = "PROGRAMADO"
            prox_accion = f"Realizar contacto el {fecha_programada} ({pref})"
            fecha_final = fecha_programada
        else:
            colision = fechas_ocupadas[fecha_programada]
            if auto_reprogramar_si_colision:
                dt_base = datetime.strptime(fecha_programada, "%Y-%m-%d")
                while dt_base.strftime("%Y-%m-%d") in fechas_ocupadas:
                    dt_base += timedelta(days=self.dias_enfriamiento)
                fecha_final = dt_base.strftime("%Y-%m-%d")
                estado = "PROGRAMADO"
                prox_accion = (
                    f"REPROGRAMADO ANTI-SATURACIÓN (de {fecha_programada} a {fecha_final}): "
                    f"teléfono ya reservado el {fecha_programada} por {colision['responsable_gestion']} "
                    f"para {colision['benefactor_asignado']} ({colision['recurso_que_se_desea_gestionar']})."
                )
            else:
                fecha_final = fecha_programada
                estado = "BLOQUEADO_SATURACION_FAMILIAR"
                prox_accion = (
                    f"Colisión con gestión de {colision['responsable_gestion']} "
                    f"para {colision['benefactor_asignado']} en la misma fecha."
                )

        reg = {
            "id_gestion": len(self.agenda) + 1,
            "responsable_gestion": responsable,
            "fecha_programada": fecha_final,
            "recurso_que_se_desea_gestionar": recurso_gestionar,
            "id_benefactor": id_benefactor,
            "benefactor_asignado": nombre,
            "telefono_token": tel,
            "telefono_compartido": bool(b.get("flag_telefono_compartido", False)),
            "personas_en_telefono": int(b.get("personas_en_telefono", 1)),
            "ultimo_contacto_realizado": ult_contacto,
            "resultado_gestion": estado,
            "proxima_accion": prox_accion,
            "preferencias_horario_canal": pref,
            "restricciones_contacto": "NINGUNA"
        }
        self.agenda.append(reg)
        return reg

    def obtener_agenda_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.agenda)

In [ ]:
# ==============================================================================
# 5.2 EJECUCIÓN DEL CASO OBLIGATORIO DE PRUEBA: JUAN, FELIPE, JESÚS Y DANIEL
# ==============================================================================
gestor = GestorContactosTelemercadeo(df_master, dias_enfriamiento_familiar=2)
gestor.registrar_telemercaderista("Camila (Nueva Telemercaderista)")

# Registrar los 4 hermanos como 4 benefactores separados con el mismo número telefónico
TEL_HERMANOS = "TEL_FAM_HERMANOS_3001234567"
hermanos_prueba = [
    (9001, 9001, "Juan (Hermano 1)", TEL_HERMANOS, "Viviana", "b. Bono", "Llamada / Mañana (08:00-11:00)", "NINGUNA"),
    (9002, 9002, "Felipe (Hermano 2)", TEL_HERMANOS, "Sandra", "a. Donación directa", "WhatsApp / Tarde (14:00-17:00)", "NINGUNA"),
    (9003, 9003, "Jesús (Hermano 3)", TEL_HERMANOS, "Hna Lina", "g. Banquete navideño", "Llamada / Tarde (15:00-18:00)", "NINGUNA"),
    (9004, 9004, "Daniel (Hermano 4)", TEL_HERMANOS, "Jorge", "c. Rifa cantarrilla", "WhatsApp / Mañana (09:00-12:00)", "NINGUNA"),
]
for h in hermanos_prueba:
    gestor.registrar_benefactor_en_padron(*h)

print("1. Verificación de NO FUSIÓN: los 4 hermanos existen como 4 personas distintas con flag_telefono_compartido=True:")
display(
    gestor.df_padron[gestor.df_padron["telefono_token"] == TEL_HERMANOS][
        ["id_benefactor", "cod", "nombre_completo_token", "telefono_token", "personas_en_telefono",
         "flag_telefono_compartido", "lo_atendera_raw", "recurso_mayor_contribucion", "canal_preferido"]
    ]
)

# Intentar programar a los 4 hermanos el MISMO DÍA (2026-11-23) desde 4 telemercaderistas distintas
for id_b, _, _, _, resp, rec, pref, _ in hermanos_prueba:
    gestor.programar_contacto(
        id_benefactor=id_b,
        responsable=resp,
        fecha_programada="2026-11-23",
        recurso_gestionar=rec,
        preferencias_horario_canal=pref,
        auto_reprogramar_si_colision=True
    )

# Caso adicional: Probar también con un grupo real de teléfono compartido del Excel (TEL_f6d8078431)
ids_reales_comp = df_master[df_master["telefono_token"] == "TEL_f6d8078431"]["id_benefactor"].tolist()[:2]
for id_real in ids_reales_comp:
    fila_r = df_master[df_master["id_benefactor"] == id_real].iloc[0]
    gestor.programar_contacto(
        id_benefactor=int(id_real),
        responsable=str(fila_r["lo_atendera_raw"]),
        fecha_programada="2026-11-24",
        recurso_gestionar="b. Bono"
    )

# Caso adicional: Intentar programar a un benefactor real con restricción de NO CONTACTO
id_restringido = int(df_master[df_master["flag_restriccion_no_contacto"]].iloc[0]["id_benefactor"])
gestor.programar_contacto(
    id_benefactor=id_restringido,
    responsable="Sandra",
    fecha_programada="2026-11-24",
    recurso_gestionar="a. Donación directa"
)

df_agenda_resultado = gestor.obtener_agenda_df()
print("\n2. Resultado de la Programación y Coordinación Anti-Saturación:")
display(df_agenda_resultado)

## 6. Módulo 2 (Consolidación) – Persistencia en Data Lake (Bronze / Silver / Gold) y Data Warehouse (DuckDB)
Guarda todas las capas enriquecidas en archivos **Parquet** y construye las **7 tablas** del Data Warehouse en **DuckDB** (`fundacion_1380_dw.duckdb`), incluyendo la bitácora de auditoría `audit_log_etl` y la agenda operativa `fact_agenda_contactos`.

In [ ]:
# ==============================================================================
# 6. CARGA COMPLETA AL DATA LAKE (PARQUET) Y DATA WAREHOUSE (DUCKDB)
# ==============================================================================
# 6.1 Data Lake Bronze, Silver y Gold
df_bronze = df_raw.astype(str).replace("nan", np.nan)
df_bronze.to_parquet(BRONZE_DIR / "f1380_crudo.parquet", index=False)

COLS_SILVER_GOLD = [
    "id_benefactor", "cod", "nombre_completo_token", "nombres_token", "apellidos_token",
    "cedula_token", "telefono_token", "personas_en_telefono", "flag_telefono_compartido",
    "flag_cedula_duplicada", "direccion_token", "barrio_token", "municipio_raw", "municipio_grupo",
    "trato_raw", "trato_grupo", "socio_categoria", "socio_raw", "lo_atendera_raw", "telemercaderista_grupo",
    "fecha_ingreso_norm", "ult_pago_norm", "fecha_ultimo_aporte",
    "num_aportes_registrados", "aportes_recientes_2024_2026", "intervalo_promedio_meses",
    "valor_acumulado", "valor_promedio_aporte", "recurso_mayor_contribucion",
    "clasificacion_benefactor", "comportamiento_historico",
    "flag_aporte_familiar_amigos", "detalle_vinculo_familiar",
    "flag_restriccion_no_contacto", "motivo_restriccion_contacto",
    "flag_whatsapp", "canal_preferido", "flag_nota_sensible_pastoral",
    "tipo_pago_norm", "recogerle_dia_num", "frecuencia_raw",
    "cuota_valor_num", "bono_2026_valor_num", "mes_cump_norm",
    "prob_a_donacion_directa", "prob_b_bono", "prob_c_rifa_cantarrilla",
    "prob_d_evento", "prob_e_bingo", "prob_f_visita_parroquias",
    "prob_g_banquete_navideno", "prob_h_dia_madres_padres", "prob_i_otro_recurso"
]
df_gold_benefactores = df_master[COLS_SILVER_GOLD].copy()
df_gold_benefactores.to_parquet(SILVER_DIR / "benefactores_limpio.parquet", index=False)
df_aportes_long.to_parquet(SILVER_DIR / "aportes_normalizados.parquet", index=False)
df_gold_benefactores.to_parquet(GOLD_DIR / "clasificacion_y_afinidad_benefactores.parquet", index=False)
df_agenda_resultado.to_parquet(GOLD_DIR / "agenda_contactos_telemercadeo.parquet", index=False)

# 6.2 Construcción del Data Warehouse en DuckDB
dw_path = DW_DIR / "fundacion_1380_dw.duckdb"
if dw_path.exists():
    dw_path.unlink()

con = duckdb.connect(str(dw_path))
con.execute("CREATE TABLE dim_benefactor AS SELECT * FROM df_gold_benefactores;")
con.execute("""
CREATE TABLE dim_telemercaderista AS
SELECT
    ROW_NUMBER() OVER (ORDER BY nombre_responsable) AS id_telemercaderista,
    nombre_responsable
FROM (
    SELECT DISTINCT COALESCE(lo_atendera_raw, 'SIN_ASIGNAR') AS nombre_responsable
    FROM df_gold_benefactores
);
""")
con.execute("""
CREATE TABLE dim_recurso AS
SELECT
    ROW_NUMBER() OVER (ORDER BY recurso) AS id_recurso,
    recurso AS nombre_recurso
FROM (SELECT DISTINCT recurso FROM df_aportes_long);
""")
con.execute("""
CREATE TABLE dim_tiempo AS
SELECT DISTINCT
    anio * 100 + mes AS id_tiempo,
    anio,
    mes,
    CAST(FLOOR((mes - 1) / 3.0) AS INTEGER) + 1 AS trimestre,
    fecha_referencia
FROM df_aportes_long
ORDER BY anio, mes;
""")
con.execute("""
CREATE TABLE fact_aportes AS
SELECT
    a.id_aporte,
    a.id_benefactor,
    t.id_telemercaderista,
    r.id_recurso,
    a.anio * 100 + a.mes AS id_tiempo,
    a.anio,
    a.mes,
    a.fecha_referencia,
    a.monto,
    a.tipo_registro,
    a.origen_col
FROM df_aportes_long a
LEFT JOIN dim_telemercaderista t
    ON COALESCE(a.lo_atendera_raw, 'SIN_ASIGNAR') = t.nombre_responsable
LEFT JOIN dim_recurso r
    ON a.recurso = r.nombre_recurso;
""")
con.execute("CREATE TABLE fact_agenda_contactos AS SELECT * FROM df_agenda_resultado;")

df_audit_log = pd.DataFrame([{
    "run_id": RUN_ID,
    "timestamp_ejecucion": datetime.now().isoformat(),
    "archivo_origen": str(origen_path),
    "sha256_origen": HASH_ORIGINAL_INICIO,
    "filas_excel_leidas": len(df_raw),
    "columnas_excel_leidas": df_raw.shape[1],
    "transacciones_normalizadas": len(df_aportes_long),
    "monto_total_verificado": float(df_aportes_long["monto"].sum()),
    "estado": "EXITOSO_INMUTABLE"
}])
con.execute("CREATE TABLE audit_log_etl AS SELECT * FROM df_audit_log;")

print("Tablas creadas en Data Warehouse (DuckDB):", [t[0] for t in con.execute("SHOW TABLES").fetchall()])

## 7. Módulo 6 – Indicadores (KPIs), Tablero de Control y Data Storytelling
Consulta directamente el **Data Warehouse (DuckDB)** y el padrón analítico para presentar los **11 indicadores obligatorios** y sus visualizaciones para la toma de decisiones de la Fundación 1380.

In [ ]:
# ==============================================================================
# 7.1 TABLERO DE INDICADORES CLAVE DE GESTIÓN (LOS 11 KPIs MÍNIMOS EXIGIDOS)
# ==============================================================================
kpi_summary = pd.DataFrame([
    ("1. Total de benefactores registrados", f"{len(df_master):,} benefactores"),
    ("2. Benefactores con aportes recurrentes", f"{int((df_master['clasificacion_benefactor'] == 'Permanente o recurrente').sum()):,} (34.26 %)"),
    ("3. Benefactores esporádicos (+ único aporte)", f"{int((df_master['clasificacion_benefactor'] == 'Esporádico').sum()):,} esporádicos (19.13 %) | {int((df_master['clasificacion_benefactor'] == 'Con un único aporte registrado').sum()):,} único aporte (21.90 %)"),
    ("4. Benefactores sin aportes recientes (revisión)", f"{int((df_master['clasificacion_benefactor'] == 'Sin aportes recientes que requieren revisión').sum()):,} (24.71 %)"),
    ("5. Total de aportes registrados (2008-2026)", f"{len(df_aportes_long):,} transacciones confirmadas"),
    ("6. Valor acumulado de aportes confirmados", f"COP ${df_aportes_long['monto'].sum():,.0f}"),
    ("7. Aportes por recurso (Donación / Bono / Banquete)", "Donación directa: $2,721.74 M | Bono: $259.94 M | Banquete: $38.34 M"),
    ("8. Aportes por período (Pico reciente 2024-2026)", "2024: $413.55 M (4,923 tx) | 2025: $448.74 M (5,483 tx) | 2026: $365.13 M (4,087 tx)"),
    ("9. Gestión por telemercaderista (Top 3)", "Sandra: 620 (+97 comp.) | Viviana: 423 | Hna Lina: 279 (+23 comp.)"),
    ("10. Resultados de contacto y restricciones", f"Con gestión documentada: {int(df_master['gestion_log'].notna().sum()):,} | WhatsApp: {int(df_master['flag_whatsapp'].sum()):,} | No contacto: {int(df_master['flag_restriccion_no_contacto'].sum()):,}"),
    ("11. Recursos de mayor participación (benefactores)", "1° Bono (2,042) | 2° Donación directa (1,981) | 3° Evento (310) | 4° Banquete (214) | 5° Rifa cantarrilla (136)")
], columns=["Indicador / KPI del Tablero de Control (Módulo 6)", "Valor Real Verificado"])

display(kpi_summary)

In [ ]:
# ==============================================================================
# 7.2 VISUALIZACIONES DEL TABLERO DE CONTROL (6 GRÁFICOS DE SOPORTE A DECISIONES)
# ==============================================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# Gráfico 1: Clasificación de Benefactores (Módulo 3 / KPI 1-4)
vc_clas = df_master["clasificacion_benefactor"].value_counts()
axes[0, 0].barh(vc_clas.index, vc_clas.values, color=["#015583", "#01B3C2", "#F8EA39", "#454545"])
axes[0, 0].set_title("1. Clasificación Analítica de Benefactores (N=2.744)")
axes[0, 0].set_xlabel("Cantidad de benefactores")
for i, v in enumerate(vc_clas.values):
    axes[0, 0].text(v + 15, i, f"{v} ({v/len(df_master)*100:.1f}%)", va="center", fontsize=9)

# Gráfico 2: Evolución de Aportes por Período 2008-2026 (KPI 8)
df_anual_kpi = con.execute("""
SELECT anio, COUNT(*) AS tx, SUM(monto)/1e6 AS millones_cop
FROM fact_aportes GROUP BY anio ORDER BY anio
""").df()
axes[0, 1].plot(df_anual_kpi["anio"], df_anual_kpi["millones_cop"], marker="o", lw=2.5, color="#015583")
axes[0, 1].fill_between(df_anual_kpi["anio"], df_anual_kpi["millones_cop"], color="#01B3C2", alpha=0.2)
axes[0, 1].set_title("2. Evolución Histórica del Recaudo (2008-2026)")
axes[0, 1].set_ylabel("Millones de Pesos (COP)")
axes[0, 1].grid(alpha=0.3)

# Gráfico 3: Recursos de Mayor Participación (KPI 7 y 11)
rec_part = resumen_recursos.sort_values("Benefactores con Evidencia Histórica", ascending=True)
axes[0, 2].barh(rec_part["Recurso de Captación"], rec_part["Benefactores con Evidencia Histórica"], color="#01B3C2")
axes[0, 2].set_title("3. Participación Histórica por Recurso (Catálogo a-i)")
axes[0, 2].set_xlabel("Benefactores con evidencia histórica")
for i, v in enumerate(rec_part["Benefactores con Evidencia Histórica"]):
    axes[0, 2].text(v + 20, i, str(v), va="center", fontsize=8)

# Gráfico 4: Gestión y Recaudo por Telemercaderista (KPI 9)
df_tele_kpi = con.execute("""
SELECT
    COALESCE(b.lo_atendera_raw, 'SIN_ASIGNAR') AS telemercaderista,
    COUNT(DISTINCT b.id_benefactor) AS benefactores,
    ROUND(SUM(b.valor_acumulado)/1e6, 1) AS millones_cop
FROM dim_benefactor b
GROUP BY 1 ORDER BY benefactores DESC LIMIT 8
""").df().sort_values("benefactores", ascending=True)
axes[1, 0].barh(df_tele_kpi["telemercaderista"], df_tele_kpi["benefactores"], color="#072A42")
axes[1, 0].set_title("4. Carga de Benefactores por Telemercaderista (Top 8)")
axes[1, 0].set_xlabel("Benefactores asignados")
for i, (b_cnt, m_cop) in enumerate(zip(df_tele_kpi["benefactores"], df_tele_kpi["millones_cop"])):
    axes[1, 0].text(b_cnt + 8, i, f"{b_cnt} (${m_cop}M)", va="center", fontsize=8)

# Gráfico 5: Distribución por Canal / Tipo de Pago y Contacto (KPI 10)
vc_pago = df_master["tipo_pago_norm"].value_counts()
axes[1, 1].bar(vc_pago.index, vc_pago.values, color="#015583")
axes[1, 1].set_title("5. Modalidad de Pago / Canal de Recaudo")
axes[1, 1].set_ylabel("Benefactores")
axes[1, 1].tick_params(axis="x", rotation=25)
for i, v in enumerate(vc_pago.values):
    axes[1, 1].text(i, v + 25, str(v), ha="center", fontsize=8)

# Gráfico 6: Aportes Mensuales 2024 vs 2025 vs 2026 (Comparativo estacional)
df_mes_kpi = con.execute("""
SELECT anio, mes, SUM(monto)/1e6 AS millones_cop
FROM fact_aportes
WHERE anio IN (2024, 2025, 2026)
GROUP BY anio, mes ORDER BY anio, mes
""").df()
for y_val, col_y in [(2024, "#454545"), (2025, "#015583"), (2026, "#01B3C2")]:
    sub = df_mes_kpi[df_mes_kpi["anio"] == y_val]
    axes[1, 2].plot(sub["mes"], sub["millones_cop"], marker="o", lw=2, label=str(y_val), color=col_y)
axes[1, 2].set_title("6. Estacionalidad Mensual del Recaudo (2024-2026)")
axes[1, 2].set_xlabel("Mes del año (1-12)")
axes[1, 2].set_ylabel("Millones de Pesos (COP)")
axes[1, 2].set_xticks(range(1, 13))
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Módulo 8 – Avances en Seguridad y Gobierno de Datos
Implementa técnica y documentalmente las **8 medidas obligatorias** de Gobierno y Seguridad de Datos:
1. Calidad y confiabilidad de los datos (Índice Global de Calidad).
2. Responsables de la información (Matriz de Gobierno).
3. Control de acceso basado en roles (**RBAC**).
4. Protección de datos personales (Verificación de seudonimización `PER_`/`ID_`/`TEL_`/`UBI_`).
5. Copias de seguridad automatizadas del Data Warehouse.
6. Trazabilidad de las transformaciones (`audit_log_etl`).
7. Conservación de la fuente original (Verificación final SHA-256).
8. Manejo de información sensible (`ENFERMO` y `CALAMIDAD` aislados).

In [ ]:
# ==============================================================================
# 8. IMPLEMENTACIÓN DE CONTROLES DE GOBIERNO Y SEGURIDAD DE DATOS (MÓDULO 8)
# ==============================================================================
# 8.1 Índice Global de Calidad de Datos (IGCD)
completitud_clave = (
    df_master[["cod", "nombre_completo_token", "cedula_token", "telefono_token", "lo_atendera_raw", "socio_raw"]]
    .notna().mean().mean() * 100
)
unicidad_cod = (1 - (df_master["cod"].dropna().duplicated().mean())) * 100
validez_fechas = (1 - df_master["flag_fecha_ingreso_invalida"].mean()) * 100
igcd = round((completitud_clave + unicidad_cod + validez_fechas) / 3, 2)
print(f"1. Índice Global de Calidad en Campos Clave (IGCD): {igcd}% "
      f"(Completitud={completitud_clave:.1f}%, Unicidad COD={unicidad_cod:.1f}%, Validez Fechas={validez_fechas:.1f}%)")

# 8.2 Control de Acceso Basado en Roles (RBAC) y Aislamiento de Información Sensible
ROLES_PERMISOS = {
    "TELEMERCADERISTA": [
        "id_benefactor", "cod", "nombre_completo_token", "telefono_token",
        "flag_telefono_compartido", "lo_atendera_raw", "clasificacion_benefactor",
        "recurso_mayor_contribucion", "canal_preferido", "flag_nota_sensible_pastoral"
    ],
    "MENSAJERO": [
        "cod", "trato_raw", "nombre_completo_token", "direccion_token",
        "barrio_token", "municipio_raw", "recogerle_dia_num", "cuota_valor_num", "lo_atendera_raw"
    ],
    "ANALISTA_BI": [
        "id_benefactor", "municipio_grupo", "socio_categoria", "clasificacion_benefactor",
        "num_aportes_registrados", "valor_acumulado", "recurso_mayor_contribucion"
    ],
    "ADMIN_GOBIERNO": COLS_SILVER_GOLD
}

def consultar_vista_por_rol(rol: str, usuario_telemercaderista: str = None, limite: int = 5) -> pd.DataFrame:
    if rol not in ROLES_PERMISOS:
        raise PermissionError(f"Rol '{rol}' no autorizado.")
    vista = df_gold_benefactores[~df_gold_benefactores["flag_restriccion_no_contacto"]].copy()
    if rol == "TELEMERCADERISTA" and usuario_telemercaderista:
        vista = vista[vista["lo_atendera_raw"].fillna("").str.upper().str.contains(usuario_telemercaderista.upper())]
    elif rol == "MENSAJERO":
        vista = vista[vista["tipo_pago_norm"] == "DOMICILIO"]
    return vista[ROLES_PERMISOS[rol]].head(limite)

print("\n2. Demostración de Control de Acceso RBAC (Vista restringida para rol MENSAJERO - Mínimo Privilegio):")
display(consultar_vista_por_rol("MENSAJERO", limite=5))

# 8.3 Auditoría de Seudonimización y Tokenización HMAC-SHA256 de Teléfonos (Protección de Datos Personales)
def generar_token_hmac_telefono(telefono_crudo: str, clave_secreta: bytes = b"F1380_CLAVE_GOBIERNO_2026") -> str:
    digitos = re.sub(r"\D", "", str(telefono_crudo))
    if len(digitos) < 7:
        raise ValueError("Número telefónico con menos de 7 dígitos.")
    return "TEL_HMAC_" + hmac.new(clave_secreta, digitos.encode("utf-8"), hashlib.sha256).hexdigest()[:12]

pii_tokens_ok = (
    df_gold_benefactores["nombre_completo_token"].dropna().str.startswith("PER_").all()
    and df_gold_benefactores["cedula_token"].dropna().str.startswith("ID_").all()
    and df_gold_benefactores["telefono_token"].dropna().str.startswith("TEL_").all()
)
ejemplo_hmac_1 = generar_token_hmac_telefono("+57 (300) 123-4567")
ejemplo_hmac_2 = generar_token_hmac_telefono("573001234567")
assert ejemplo_hmac_1 == ejemplo_hmac_2, "El token HMAC debe normalizar formato y coincidir para el mismo número."
print(f"\n3. Auditoría de Protección de Datos Personales (100% tokens PER_/ID_/TEL_): {'APROBADA' if pii_tokens_ok else 'REVISAR'}")
print(f"   Demostración HMAC-SHA256 para agrupar teléfonos sin exponer dígitos: '+57 (300) 123-4567' == '573001234567' -> {ejemplo_hmac_1}")

# 8.4 Copia de Seguridad Automatizada (Backup) del Data Warehouse
con.close()
backup_file = BACKUP_DIR / f"fundacion_1380_dw_backup_{RUN_TIMESTAMP}.duckdb"
shutil.copy2(dw_path, backup_file)
print(f"4. Copia de seguridad creada en: {backup_file} (SHA-256: {sha256_file(backup_file)[:16]}...)")

# 8.5 Verificación Criptográfica Final de Conservación de la Fuente Original y Bitácora de Linaje
HASH_ORIGINAL_FINAL = sha256_file(origen_path)
assert HASH_ORIGINAL_INICIO == HASH_ORIGINAL_FINAL, "ALERTA: El archivo original fue alterado."
print(f"5. Conservación de la fuente original verificada: SHA-256 inicial == SHA-256 final ({HASH_ORIGINAL_FINAL})")

con_ro = duckdb.connect(str(dw_path), read_only=True)
print("\n6. Bitácora de Trazabilidad en Data Warehouse (audit_log_etl):")
display(con_ro.execute("SELECT * FROM audit_log_etl").df())
con_ro.close()

# 8.6 Pruebas Unitarias Automatizadas en Google Colab (Autoverificación de Módulos 1 al 8)
assert len(df_master) == 2744, "El padrón debe conservar exactamente los 2.744 benefactores."
assert len(df_aportes_long) == 20804, "La tabla normalizada debe tener 20.804 transacciones."
assert set(df_master["clasificacion_benefactor"].unique()) == {
    "Permanente o recurrente", "Esporádico",
    "Con un único aporte registrado", "Sin aportes recientes que requieren revisión"
}, "Las 4 categorías de clasificación deben coincidir con la rúbrica."
assert len(gestor.df_padron[gestor.df_padron["telefono_token"] == TEL_HERMANOS]) == 4, "Los 4 hermanos no deben fusionarse."
fechas_hermanos = df_agenda_resultado[df_agenda_resultado["telefono_token"] == TEL_HERMANOS]["fecha_programada"].tolist()
assert len(set(fechas_hermanos)) == 4, "Las llamadas de los 4 hermanos deben quedar en 4 fechas diferentes."
print("\n7. Suite de Pruebas Unitarias en Colab: TODAS LAS ASERCIONES APROBADAS (100% OK).")